# 🥈 02 — Dhara: Silver Layer — Feature Engineering from Real Data

This is where we turn raw schedules into ML-ready features. **All features are derived from real data:**

| Feature | Source | Domain justification |
|---------|--------|---------------------|
| `stop_seq` | schedules | Later stops → more accumulated delay |
| `total_stops` | schedules | More stops = more dwell variance |
| `cumulative_travel_min` | schedules (dep-origin) | How far into the journey |
| `dwell_min` | schedules (dep-arr) | Long dwell = major junction, more variance |
| `scheduled_hour` | schedules | Peak hours drive congestion |
| `day_of_journey` | schedules | Multi-day trains accumulate drift |
| `train_type` | trains.json | Express vs Passenger vs DEMU differ in priority |
| `zone` | stations.json | NR zones have more fog delay |
| `state_pm25` | AQ CSV | Fog proxy — real PM2.5 for the station's state |
| `total_distance_km` | trains.json | Longer journeys = more variance |
| `lat`, `lon` | stations.json | Geographic cluster features |

In [0]:
%sql
USE CATALOG setu_rail;

## Step 1 — Build zone → state → PM2.5 lookup table

In [0]:
from pyspark.sql import Row

# Zone-to-state mapping (based on Indian Railways zonal headquarters)
# PM2.5 is joined from AQ data for the state each zone primarily covers.
# This gives us a real, defensible feature from real data.
zone_state_map = [
    ("NR",   "DELHI"),
    ("NCR",  "UTTAR PRADESH"),
    ("NWR",  "RAJASTHAN"),
    ("NER",  "UTTAR PRADESH"),
    ("NFR",  "ASSAM"),
    ("ECR",  "BIHAR"),
    ("ER",   "WEST BENGAL"),
    ("SER",  "JHARKHAND"),
    ("ECoR", "ODISHA"),
    ("SCR",  "TELANGANA"),
    ("SECR", "CHHATTISGARH"),
    ("CR",   "MAHARASHTRA"),
    ("WR",   "GUJARAT"),
    ("WCR",  "MADHYA PRADESH"),
    ("SR",   "TAMIL NADU"),
    ("SWR",  "KARNATAKA"),
    ("KR",   "KERALA"),
]

zone_state_df = spark.createDataFrame(
    [Row(zone=z, state_for_aq=s) for z, s in zone_state_map]
)

(zone_state_df.write
    .format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("setu_rail.silver.zone_state_map"))

print("✅ silver.zone_state_map written")

## Step 2 — Core feature engineering SQL

This is our most important cell. We use window functions (Spark SQL) to compute cumulative travel time and stop sequence — real ML features derived from real schedule data.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE setu_rail.silver.schedule_features
USING DELTA
AS
WITH
-- Step A: Parse all times to absolute minutes (day*1440 + H*60 + M)
sched_parsed AS (
    SELECT
        s.train_number                                                      AS train_no,
        s.train_name,
        s.station_code,
        s.station_name,
        s.day,
        s.arrival,
        s.departure,
        -- Parse departure to integer minutes (absolute within journey)
        CASE WHEN s.departure IS NOT NULL AND s.departure != 'None' THEN
            COALESCE(s.day, 1) * 1440
            + CAST(SPLIT(s.departure, ':')[0] AS INT) * 60
            + CAST(SPLIT(s.departure, ':')[1] AS INT)
        END                                                                 AS dep_abs_min,
        -- Parse arrival to integer minutes
        CASE WHEN s.arrival IS NOT NULL AND s.arrival != 'None' THEN
            COALESCE(s.day, 1) * 1440
            + CAST(SPLIT(s.arrival, ':')[0] AS INT) * 60
            + CAST(SPLIT(s.arrival, ':')[1] AS INT)
        END                                                                 AS arr_abs_min,
        CASE WHEN s.departure IS NOT NULL THEN
            CAST(SPLIT(s.departure, ':')[0] AS INT)
        END                                                                 AS scheduled_hour
    FROM setu_rail.bronze.schedules s
    WHERE s.departure IS NOT NULL
),

-- Step B: Add stop sequence and origin departure time using window functions
sched_seq AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY train_no ORDER BY dep_abs_min ASC NULLS LAST
        ) - 1                                                               AS stop_seq,
        COUNT(*) OVER (PARTITION BY train_no)                              AS total_stops,
        FIRST(dep_abs_min) OVER (
            PARTITION BY train_no ORDER BY dep_abs_min ASC NULLS LAST
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        )                                                                   AS origin_dep_min
    FROM sched_parsed
    WHERE dep_abs_min IS NOT NULL
),

-- Step C: Compute derived features
sched_features AS (
    SELECT
        train_no, train_name, station_code, station_name,
        day                                                                 AS journey_day,
        arrival, departure, scheduled_hour,
        stop_seq, total_stops,
        dep_abs_min - origin_dep_min                                        AS cumulative_travel_min,
        GREATEST(0,
            CASE WHEN arr_abs_min IS NOT NULL AND dep_abs_min IS NOT NULL AND dep_abs_min > arr_abs_min
                 THEN dep_abs_min - arr_abs_min
                 ELSE 0
            END
        )                                                                   AS dwell_min,
        CASE WHEN scheduled_hour BETWEEN 6 AND 10
             OR  scheduled_hour BETWEEN 17 AND 21
             THEN 1 ELSE 0 END                                              AS is_peak_hour,
        CASE WHEN total_stops > 100 THEN 'long'
             WHEN total_stops > 30  THEN 'medium'
             ELSE 'short' END                                               AS route_length_cat
    FROM sched_seq
),

-- Step D: Join with trains metadata
with_train AS (
    SELECT
        f.*,
        t.train_type, t.zone AS train_zone, t.total_distance_km,
        t.duration_hours, t.has_sleeper
    FROM sched_features f
    LEFT JOIN setu_rail.bronze.trains t ON t.train_no = f.train_no
),

-- Step E: Join with stations metadata (lat/lon/state/zone)
with_station AS (
    SELECT
        w.*,
        st.state,
        COALESCE(st.zone, w.train_zone)                                     AS zone,
        st.latitude, st.longitude
    FROM with_train w
    LEFT JOIN setu_rail.bronze.stations st ON st.station_code = w.station_code
),

-- Step F: Join with AQ data via zone → state mapping
with_aq AS (
    SELECT
        ws.*,
        COALESCE(aq.pm25, 60.0)                                             AS pm25,
        COALESCE(aq.no2,  30.0)                                             AS no2,
        COALESCE(aq.pm10, 80.0)                                             AS pm10
    FROM with_station ws
    LEFT JOIN setu_rail.silver.zone_state_map zsm ON zsm.zone = ws.zone
    LEFT JOIN setu_rail.bronze.air_quality aq
        ON UPPER(aq.state) = UPPER(zsm.state_for_aq)
)

SELECT * FROM with_aq
""")

count = spark.table("setu_rail.silver.schedule_features").count()
print(f"✅ silver.schedule_features: {count:,} rows")

In [0]:
%sql
-- Sanity check: look at feature ranges
SELECT
    COUNT(*)                                    AS total_records,
    COUNT(DISTINCT train_no)                    AS unique_trains,
    COUNT(DISTINCT station_code)                AS unique_stations,
    ROUND(AVG(stop_seq), 1)                     AS avg_stop_seq,
    MAX(total_stops)                            AS max_stops,
    ROUND(AVG(cumulative_travel_min), 1)        AS avg_cumulative_min,
    ROUND(AVG(dwell_min), 2)                    AS avg_dwell_min,
    ROUND(AVG(pm25), 1)                         AS avg_pm25
FROM setu_rail.silver.schedule_features;

In [0]:
%sql
-- Showcase: PM2.5 by zone — real data driving a real feature
SELECT zone,
       ROUND(AVG(pm25), 1) AS avg_pm25,
       COUNT(DISTINCT train_no) AS trains,
       COUNT(*) AS stop_records
FROM   setu_rail.silver.schedule_features
WHERE  zone IS NOT NULL
GROUP  BY zone
ORDER  BY avg_pm25 DESC;

✅ **Next:** `01_synthesize_delays.ipynb` → builds gold.features_delay_ml with the label